<div align="center">
  <a href="https://pageindex.ai/developer">
    <img src="https://github.com/user-attachments/assets/bae02956-6c4e-4a0b-adea-257b0be4aaa1" width="100%" alt="PageIndex — Vectorless, Reasoning-based RAG">
  </a>
  <p>
    <a href="https://pageindex.ai/developer">🌐 Website</a> &nbsp; · &nbsp;
    <a href="https://developer.pageindex.ai/">☁️ Cloud</a> &nbsp; · &nbsp;
    <a href="https://docs.pageindex.ai/">📖 Docs</a> &nbsp; · &nbsp;
    <a href="https://github.com/VectifyAI/PageIndex">⭐ GitHub</a> &nbsp; · &nbsp;
    <a href="https://pageindex.ai/contact">✉️ Contact</a>
  </p>
</div>

---

# PageIndex Document Metadata

**Tag your documents when you upload them, then ask by tag.**

Tags carry what a file does not say about itself: which team owns it, what kind of document it is, which year it belongs to. This notebook demonstrates how to attach **your own metadata** to documents on PageIndex Cloud and use it in chat.

We upload four documents, each tagged with its `type`, `team`, and `year`. The tags come back with every document, and the chat agent sees them when it looks through your library, so a question can name documents by their tags instead of their file names.

1. **Tag on upload** — Pass a `metadata` dict to `submit_document()`.
2. **Read the tags back** — Get them from `get_document()`.
3. **Ask by tag** — Name documents by their tags in a chat question.

**Before you start:** Have your [PageIndex API key](https://developer.pageindex.ai/api-keys) and OpenAI API key ready. This demo uploads four small PDFs to your PageIndex Cloud library.


## 01 · Set up

Install the latest PageIndex SDK. Requests downloads the demo documents.


In [ ]:
!pip install -U pageindex requests

### Configure your client

Set your API keys and choose your chat `MODEL`. The client indexes documents on PageIndex Cloud, and your chosen model answers with PageIndex Cloud's retrieval tools.


In [2]:
import os
from pageindex import PageIndexClient

os.environ["PAGEINDEX_API_KEY"] = "Your PageIndex API key here"
os.environ["OPENAI_API_KEY"] = "Your OpenAI API key here"

MODEL = "gpt-5.6-luna"  # gpt-5.6-sol / gpt-5.6-terra / gpt-5.6-luna

client = PageIndexClient(
    index="cloud",
    chat=MODEL,
)

## 02 · Tag documents on upload

Download four PDFs from the PageIndex repository and submit each one with its tags. `team` is something no file states about itself; only the tag knows it. `wait=True` returns once the document is ready.


In [3]:
import requests

BASE_URL = "https://raw.githubusercontent.com/VectifyAI/PageIndex/main/examples/documents/"
DOCS = {
    "q1-fy25-earnings.pdf": {"type": "earnings_release", "team": "finance", "year": 2025},
    "Regulation Best Interest_Interpretive release.pdf": {"type": "regulation", "team": "legal", "year": 2019},
    "earthmover.pdf": {"type": "paper", "team": "research", "year": 2013},
    "attention-residuals.pdf": {"type": "paper", "team": "research", "year": 2026},
}

doc_ids = {}
for file_name, metadata in DOCS.items():
    response = requests.get(BASE_URL + file_name, timeout=60)
    response.raise_for_status()
    with open(file_name, "wb") as f:
        f.write(response.content)

    doc_ids[file_name] = client.submit_document(file_name, wait=True, metadata=metadata)["doc_id"]
    print(f"{file_name}: {doc_ids[file_name]}")

q1-fy25-earnings.pdf: pi-cmvhw60k2005p09nzoonptd2q
Regulation Best Interest_Interpretive release.pdf: pi-cmvhw7hz600760anzjumejz6e
earthmover.pdf: pi-cmvhw8byd005g0cnzszgdoaes
attention-residuals.pdf: pi-cmvhw94zp00770anzrvi6u022


## 03 · Read the tags back

The tags come back under `metadata` from `get_document()`, and on every document from `list_documents()`. Metadata is set at submission; to change it, delete the document and submit it again.


In [ ]:
for file_name, doc_id in doc_ids.items():
    print(f"{file_name}: {client.get_document(doc_id)['metadata']}")

## 04 · Ask by tag

Ask about "the finance team's earnings release" without naming the file. The agent finds the document by its tags, then reads its pages. The output streams the answer alongside retrieval tool calls.


In [5]:
messages = [{"role": "user", "content": "What revenue did the finance team's 2025 earnings release report?"}]

for event in client.chat(messages, stream=True, protocol="responses"):
    if event["type"] == "response.output_text.delta":
        print(event["delta"], end="", flush=True)
    elif event["type"] == "response.output_item.done":
        item = event["item"]
        if item.get("type") == "function_call":
            print(f"\n[Tool call] {item['name']}({item.get('arguments', '')})", flush=True)


[Tool call] get_folder_structure({})

[Tool call] browse_documents({"folder_id":"root","recursive":false,"sort":"relevance","query":"finance team's 2025 earnings release revenue","limit":10})

[Tool call] get_document({"doc_name":"q1-fy25-earnings.pdf","folder_id":"root","wait_for_completion":false})

[Tool call] get_document_structure({"doc_name":"q1-fy25-earnings.pdf","folder_id":"root","part":1,"wait_for_completion":false})

[Tool call] get_page_content({"doc_name":"q1-fy25-earnings.pdf","folder_id":"root","pages":"1-3","wait_for_completion":false})
The finance team’s 2025 Q1 earnings release reported **$24.69 billion in revenue**, up **5%** from **$23.549 billion** in Q1 fiscal 2024.

Tags also answer questions on their own. List the research team's papers: the agent reads the team and year from the tags, not from the papers.


In [6]:
messages = [{"role": "user", "content": "Which papers does the research team have? List each with its year, oldest first."}]

for event in client.chat(messages, stream=True, protocol="responses"):
    if event["type"] == "response.output_text.delta":
        print(event["delta"], end="", flush=True)
    elif event["type"] == "response.output_item.done":
        item = event["item"]
        if item.get("type") == "function_call":
            print(f"\n[Tool call] {item['name']}({item.get('arguments', '')})", flush=True)


[Tool call] get_folder_structure({})

[Tool call] browse_documents({"folder_id":"root","recursive":false,"sort":"relevance","query":"research team papers","limit":50})

[Tool call] get_metadata_schema({})
The research team has these papers, oldest first:

1. **earthmover.pdf** — **2013**
2. **attention-residuals.pdf** — **2026**

### Metadata rules

Metadata is a flat dict of `str`, `int`, `float`, or `bool` values, with keys that start with a letter and contain only letters, digits, and underscores. Its serialized size is at most 8 KB, and PageIndex Cloud rejects metadata that breaks a rule with `PageIndexAPIError`. See the [metadata documentation](https://docs.pageindex.ai/sdk/documents#metadata) for details.


---

<div align="center">
  <br>
  <p>
    <a href="https://docs.pageindex.ai/"><b>Upload your documents</b></a>
    &nbsp;&nbsp; · &nbsp;&nbsp;
    <a href="https://docs.pageindex.ai/sdk/agents"><b>Integrate with your agent</b></a>
    &nbsp;&nbsp; · &nbsp;&nbsp;
    <a href="https://github.com/VectifyAI/PageIndex"><b>Star PageIndex on GitHub</b></a>
  </p>
  <br>
  <a href="https://pageindex.ai">
    <img src="https://github.com/user-attachments/assets/e1c677f2-b590-4ffc-859d-e8cc7d794bdf" width="144" height="40" alt="PageIndex">
  </a>
</div>
